In [0]:
spark.sql("USE CATALOG hdb_resale_prices")
spark.sql("USE SCHEMA bronze")


In [0]:
from pyspark.sql import functions as F

volume_path = "/Volumes/hdb_resale_prices/bronze/raw_datasets"

# file mapping
files = {
    "90_99" : "/Volumes/hdb_resale_prices/bronze/raw_datasets/Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",
    "00_feb12" : "/Volumes/hdb_resale_prices/bronze/raw_datasets/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
    "jan15_dec16" : "/Volumes/hdb_resale_prices/bronze/raw_datasets/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
    "mar12_dec14" : "/Volumes/hdb_resale_prices/bronze/raw_datasets/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
    "jan17_onward" : "/Volumes/hdb_resale_prices/bronze/raw_datasets/Resale flat prices based on registration date from Jan-2017 onwards.csv",
}

# Create list of df metadata
dataframes = []
for era, filename in files.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(filename)
        .withColumn("source_era", F.lit(era))
        .withColumn("source_file", F.lit(filename))
        .withColumn("ingested_at", F.current_timestamp())
    )
    dataframes.append((era, filename, df))


In [0]:
combined = dataframes[0][2]

# need allw missing col cause new column present in later dataset
for era, filename, df in dataframes[1:]:

    # combine dfs tgt by col name
    combined = combined.unionByName(df, allowMissingColumns=True)

# replace existing table data with actual source records
combined.write.mode("overwrite").saveAsTable("hdb_resale_prices.bronze.raw_datasets")


In [0]:
import uuid

# generate unique id
batch_id = str(uuid.uuid4())

log_rows = []
for era, filename, df in dataframes:
    row_count = df.count()

    log_row = (
        batch_id,
        era,
        filename,
        row_count,
    )

    log_rows.append(log_row)

# chnge list to spark df
log_df = spark.createDataFrame(log_rows, ["batch_id", "source_era", "source_file", "row_count"]) \
    .withColumn("loaded_at", F.current_timestamp())

# control table 
log_df.write.mode("append").saveAsTable("hdb_resale_prices.bronze.ctrl_ingestion_log")